# Семинар 06. ООП и принципы SOLID


## Цели

После семинара вы сможете:

- распознавать нарушения принципов SOLID в небольших примерах;
- разделять ответственности между классами;
- проектировать расширяемые зависимости через абстракции.

## Перед началом

Нужны классы, наследование, композиция и абстрактные базовые классы.


## Что такое SOLID

SOLID — пять принципов проектирования модулей и зависимостей:

- **S** — Single Responsibility Principle, принцип единственной ответственности;
- **O** — Open/Closed Principle, принцип открытости/закрытости;
- **L** — Liskov Substitution Principle, принцип подстановки Лисков;
- **I** — Interface Segregation Principle, принцип разделения интерфейса;
- **D** — Dependency Inversion Principle, принцип инверсии зависимостей.

Они нужны, чтобы изменение одного правила не заставляло переписывать полпроекта и гадать, что отвалится следом. Если модуль знает обо всём, делает всё и напрямую создаёт все свои зависимости, это не простота, а отложенный счёт за каждое следующее изменение.

SOLID — не закон природы и не повод заранее строить двадцать интерфейсов вокруг трёх функций. Принципы применяют там, где уже видна ось изменений. Бессмысленная абстракция портит код не хуже, чем отсутствие нужной абстракции.

## S: Single Responsibility Principle

У модуля должна быть одна причина для изменения — один источник требований, или актор. «Класс работает с пользователем» ещё не одна ответственность: правила расчёта скидки меняет бизнес, формат JSON — контракт API, а сохранение в PostgreSQL — инфраструктура. Если всё это засунуто в `UserService`, класс меняется по трём независимым причинам и связывает несвязанные решения.

Простой тест: если в описании класса естественно появляется несколько независимых союзов «и», класс, скорее всего, делает лишнее. Делить код на классы по одному методу тоже не надо: цель SRP — высокая связность внутри модуля и слабая связанность между разными причинами изменения.

## O: Open/Closed Principle

Стабильный модуль должен позволять добавить ожидаемый вариант поведения без переписывания уже проверенной основной логики. Расширение может происходить через композицию, функцию-стратегию, конфигурацию, регистрацию обработчика или наследование. Создавать подкласс на каждое изменение принцип не требует.

OCP работает только относительно конкретной оси изменений. Невозможно закрыть модуль от всех будущих изменений, потому что будущее не обязано уважать наши догадки. Сначала находят реально меняющуюся часть, затем ставят границу абстракции именно там.

## L: Liskov Substitution Principle

Объект подтипа должен заменять объект базового типа, не ломая обоснованные ожидания клиентского кода. Подтип не должен усиливать предусловия, ослаблять постусловия или нарушать инварианты базового типа.

Сам факт `class A(B)` ничего не гарантирует. Он только сообщает интерпретатору о наследовании. Если `A` меняет смысл операций `B`, это плохой подтип, сколько бы методов он ни унаследовал. Классический пример — изменяемые прямоугольник и квадрат:


In [ ]:
class Rectangle:
    def __init__(self, width: int, height: int) -> None:
        self.width = width
        self.height = height
    
    def area(self) -> int:
        return self.width * self.height

    def set_width(self, width: int) -> None:
        self.width = width

    def set_height(self, height: int) -> None:
        self.height = height


class Square(Rectangle):
    def __init__(self, side: int) -> None:
        super().__init__(side, side)
    
    def set_width(self, width: int) -> None:
        self.width = width
        self.height = width

    def set_height(self, height: int) -> None:
        self.width = height
        self.height = height


def resize_to_5_by_4(rectangle: Rectangle) -> None:
    rectangle.set_width(5)
    rectangle.set_height(4)
    assert rectangle.area() == 20


resize_to_5_by_4(Rectangle(3, 4))
# Square нарушает ожидание независимого изменения сторон: assertion упадёт.
# resize_to_5_by_4(Square(3))


Квадрат математически является прямоугольником, но изменяемый `Square` не является поведенческим подтипом данного изменяемого `Rectangle`. Клиент вправе ожидать, что `set_height()` не меняет ширину. Нормальное решение — не наследовать эти классы друг от друга: сделать их независимыми типами либо дать обоим общий неизменяемый интерфейс фигуры с методом `area()`.

[Исходная работа Барбары Лисков о подстановке и иерархиях типов](https://www.cs.tufts.edu/~nr/cs257/archive/barbara-liskov/data-abstraction-and-hierarchy.pdf).

## I: Interface Segregation Principle

Клиент не должен зависеть от методов, которые ему не нужны. Толстый интерфейс `Animal` с методами `fly()`, `walk()` и `swim()` заставит собаку изображать полёт, а птицу — подводную лодку. Заглушка `raise NotImplementedError` не исправляет дизайн: она честно сообщает, что контракт врёт.

Интерфейс делят по потребностям клиентов. Один класс может реализовать несколько узких интерфейсов:

```python
from abc import ABC, abstractmethod


class Flyable(ABC):
    @abstractmethod
    def fly(self) -> None:
        ...


class Bird(Flyable):
    def fly(self) -> None:
        print("I'm flying")


class Airplane(Flyable):
    def fly(self) -> None:
        print("I'm flying with engines")


class Walkable(ABC):
    @abstractmethod
    def walk(self) -> None:
        ...


class Dog(Walkable):
    def walk(self) -> None:
        print("I'm walking")


class Duck(Flyable, Walkable):
    def fly(self) -> None:
        print("I'm flying")

    def walk(self) -> None:
        print("I'm walking")
```

## D: Dependency Inversion Principle

- Модуль с бизнес-правилом не должен зависеть от конкретной базы, HTTP-клиента или устройства. И политика, и детали зависят от абстракции.
- Абстракцию формулируют в терминах потребности высокоуровневого кода. Деталь реализует этот контракт, а не диктует приложению собственный API.

Если `FlyablesManager` сам создаёт `Bird()` и `Airplane()`, он жёстко прибит к этим классам. Передача зависимостей снаружи разрывает сцепку:

```python
from collections.abc import Iterable


class FlyablesManager:
    def __init__(self, flyables: Iterable[Flyable] = ()) -> None:
        self._flyables = list(flyables)

    def add_flyable(self, flyable: Flyable) -> None:
        self._flyables.append(flyable)

    def fly_all(self) -> None:
        for flyable in self._flyables:
            flyable.fly()


manager = FlyablesManager([Bird(), Airplane()])
manager.fly_all()
```

Передача зависимости через конструктор — dependency injection. Это техника, а не сам DIP: можно внедрить десять конкретных инфраструктурных классов и всё равно сохранить плохое направление зависимостей. Суть DIP — кто определяет контракт и от чего зависит важная логика.


## Как принципы связаны

| Проблема в коде | Что бьёт тревогу | Что обычно исправляют |
|---|---|---|
| Один класс меняется из-за бизнес-правил, формата API и базы | SRP | разделяют независимые ответственности |
| Для каждого нового варианта приходится лезть в старую цепочку `if/elif` | OCP | выделяют реальную ось изменений и точку расширения |
| Подкласс требует особых проверок или ломает ожидания базы | LSP | исправляют контракт, композицию или иерархию |
| Реализации содержат заглушки для ненужных методов | ISP | делят толстый интерфейс по потребностям клиентов |
| Бизнес-логика напрямую создаёт инфраструктурные детали | DIP | разворачивают зависимость через контракт и внедрение реализации |

Принципы перекрываются. Это нормально: плохая граница редко нарушает ровно одну букву. Важно не назвать нарушение, а сделать следующее изменение локальным и предсказуемым. Если после «улучшения по SOLID» код стал длиннее, а изменение по-прежнему размазано по проекту, улучшения не произошло.


## Самопроверка

1. Почему «один метод на класс» не является правильным толкованием SRP?
2. От каких изменений модуль вообще может быть закрыт согласно OCP?
3. Почему наследование `Square(Rectangle)` не доказывает выполнение LSP?
4. Что не так с реализацией интерфейса через `raise NotImplementedError` для половины методов?
5. Чем DIP отличается от передачи любого объекта через конструктор?
6. В какой момент новая абстракция помогает, а в какой только увеличивает объём кода?


## Итоги

- SRP отделяет причины изменения, а не считает методы.
- OCP защищает стабильную логику от ожидаемого варианта изменений, но не от любого будущего.
- LSP проверяет поведение подтипа с точки зрения клиента, а не форму иерархии.
- ISP не заставляет клиентов и реализации зависеть от ненужных операций.
- DIP направляет зависимость от деталей к контракту, нужному бизнес-логике.
- SOLID полезен как набор диагностических принципов; механическое следование аббревиатуре производит архитектурный мусор.


## Задание 1. Blackjack (до 2 баллов)
Нам понадобится
```python
import abc


class Card:
    def __init__(self, number):
        # todo - проверить, что номер от 0 до 51 включительно
        self._number = number
    
    def __str__(self):
        # здесь выводим строку из номинала карты и масти, например 10❤️. Считаем, что порядок карт по умолчанию такой: 
        # сначала все пики (от 2 до A),потом все крести, потом бубны и черви соответственно.
        pass


class CardValueManager(abc.ABC):
    @abc.abstractmethod
    def get_value(self, card: Card) -> int:
        """
        Возвращает значение карты. В простейшей реализации этого интерфейса, можно предположить, 
        что карты с цифрами стоят столько, сколько на них написано, все картинки по 10, туз 11. Не возбраняется модифицировать этот интерфейс таким образом, 
        чтобы он допускал возможность возврата нескольких возможных значений. У всех карт, кроме тузов, по правилам, описанным выше, у тузов список из [1, 11]
        """
        pass

class Deck:
    # здесь случайно перемешанная колода карт хранится где-то, где мы запомним в конструкторе.
    
    def next_card(self) -> Card:
        # здесь возвращаем следующую карту из колоды. Даем знать, если карты закончились. Для этого использовать генератор
        pass

class Player:
    # здесь нужно знать его статус: выиграл он, проиграл или не известно, сумму очков в руке и список карт. Нужна возможность добавить карту в руку (из колоды).
    pass

class GameEngine:
    """
    Здесь нужно реализовать следующую игровую механику. 
    Есть два игрока: Игрок и Дилер. В начале игры раздается две карты игроку (пользователь их видит) и одна карта дилеру. Пользователь ее видит. Начинается ход игрока.
    Пока идет ход игрока, пользователь, видя свои карты, может принять решение, продолжать игру или нет. 
    Если игрок выбирает продолжать, ему выдается еще одна карта и пересчитывается сумма очков.
    Если сумма очков игрока превышает 21, то он немедленно проигрывает. Если сумма очков игрока равна 21, то он немедленно выигрывает. 
    Иначе пользователь может принять решение: продолжать игру или передать ход дилеру.

    Во время хода дилера, пока сумма очков дилера меньше 17, то он берет карту. При обработке очередной карты, если сумма очков превышает 21, то он проигрывает. Если сумма очков равна 21, то он выигрывает.
    Иначе сравнивается сумма очков игрока и дилера, у кого больше, тот и выиграл. Ничья тоже возможна
    """
    pass

```

Каждый класс помещаем в отдельный файл.



**Оценивание и критерии проверки**

- 1 балл: реализованы `Card`, `CardValueManager` и `Deck`, проверяются границы номера карты и окончание колоды;
- 2 балла: дополнительно реализованы `Player` и `GameEngine`, включая туз как 1 или 11, перебор дилера до 17, победу, поражение и ничью;
- каждый класс находится в отдельном модуле, а правила игры покрыты тестами без пользовательского ввода внутри бизнес-логики.
